# Сопоставление подписантов из JSON со справочником CSV
Настройте пути, два ключа JSON и название столбца CSV в начале следующей ячейки. Используются биграммы и триграммы для фамилии (70%) и сравнение инициалов (30%) со штрафом за несовпадение.

Зависимости: `pandas`, `openpyxl`. Установка при необходимости: `pip install pandas openpyxl`.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

# ===== НАСТРОЙКИ =====
JSON_DIR = Path("input_json")
CSV_FILE = "signers.csv"
OUTPUT_FILE = "result.xlsx"
JSON_KEYS = ["signer1", "signer2"]  # Укажите два ключа JSON
CSV_COLUMN = "ФИО"                 # Укажите столбец CSV


def parse_fio(text):
    text = str(text).strip().lower().replace("ё", "е")
    parts = re.findall(r"[а-яa-z]+", text)
    if not parts:
        return "", ""
    return parts[0], "".join(p[0] for p in parts[1:3])


def ngrams(text, n):
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i+n] for i in range(len(text)-n+1)}


def ngram_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    scores = []
    for n in (2, 3):
        x, y = ngrams(a, n), ngrams(b, n)
        if x and y:
            scores.append(len(x & y) / len(x | y))
    return sum(scores) / len(scores) if scores else 0


def initials_similarity(a, b):
    if not a or not b:
        return 0
    if a == b:
        return 1
    matches = sum(x == y for x, y in zip(a, b))
    return matches / max(len(a), len(b))


def similarity(fio1, fio2):
    surname1, initials1 = parse_fio(fio1)
    surname2, initials2 = parse_fio(fio2)
    if not surname1 or not surname2:
        return 0
    surname_score = ngram_similarity(surname1, surname2)
    initials_score = initials_similarity(initials1, initials2)
    if not initials1 or not initials2:
        return surname_score * 0.7
    score = surname_score * 0.7 + initials_score * 0.3
    if initials1 != initials2:
        score *= 0.7
    return score


df = pd.read_csv(CSV_FILE, sep=";", encoding="utf-8-sig")
signers = df[CSV_COLUMN].dropna().astype(str).tolist()
results = []

for file in JSON_DIR.glob("*.json"):
    with open(file, encoding="utf-8") as f:
        data = json.load(f)
    for key in JSON_KEYS:
        original = data.get(key)
        if not isinstance(original, str) or not original.strip():
            continue
        matches = [(signer, similarity(original, signer)) for signer in signers]
        best_name, best_score = max(matches, key=lambda x: x[1], default=("", 0))
        results.append({
            "Файл": file.name,
            "Ключ JSON": key,
            "Исходное ФИО": original,
            "Лучшее совпадение CSV": best_name,
            "Сходство, %": round(best_score * 100, 2),
        })

pd.DataFrame(results, columns=["Файл", "Ключ JSON", "Исходное ФИО", "Лучшее совпадение CSV", "Сходство, %"]).to_excel(OUTPUT_FILE, index=False)
print(f"Готово! Обработано записей: {len(results)}")
print(f"Результат: {OUTPUT_FILE}")
